## Testes para aprendizado em criação das funções

### Toy example: carteira com PETR4, VALE3 e ITUB4

Roteiro do modelo Black-Litterman, uma etapa por vez:

1. **Dados**: preços (yfinance), market cap (yfinance) e taxa livre de risco (CDI, BCB)
2. **Retornos e covariância** $\Sigma$
3. **Prior**: pesos de mercado $w_{mkt}$, aversão ao risco $\delta$ e retornos de equilíbrio $\Pi = \delta \Sigma w_{mkt}$
4. **Views**: matrizes $P$, $Q$, $\Omega$ e o escalar $\tau$
5. **Posterior**: retornos combinados $\mu_{BL}$
6. **Otimização**: pesos finais da carteira

## Etapa 1: Coleta de dados

São necessários três insumos:

| Insumo | Fonte | Para que serve |
|---|---|---|
| Preços de fechamento ajustados | yfinance | retornos e covariância $\Sigma$ |
| Market cap | yfinance | pesos de equilíbrio $w_{mkt}$ (o "mercado" no prior) |
| CDI diário | BCB, SGS série 12 | taxa livre de risco $r_f$ (retorno em excesso e $\delta$) |

Usamos uma janela fixa de 5 anos para que os dois computadores obtenham o mesmo resultado.

In [ ]:
import numpy as np
import pandas as pd
import requests
import yfinance as yf

TICKERS = ["PETR4", "VALE3", "ITUB4"]
INICIO, FIM = "2021-11-01", "2026-11-01"   # fim exclusivo no yfinance

### 1.1 Preços

- O yfinance exige o sufixo `.SA` para ativos da B3.
- `auto_adjust=True` ajusta o preço por dividendos e desdobramentos. Sem isso, a data-ex de um dividendo aparece como uma queda falsa no retorno.

In [ ]:
precos = yf.download(
    [t + ".SA" for t in TICKERS],
    start=INICIO, end=FIM, auto_adjust=True, progress=False,
)["Close"]

precos.columns = precos.columns.str.replace(".SA", "")   # volta ao ticker puro
precos = precos[TICKERS].dropna()                         # ordem fixa das colunas

print(precos.shape)
precos.tail()

### 1.2 Market cap

O Black-Litterman parte do princípio de que **a carteira de mercado é a ótima** quando não há views. Por isso os pesos de equilíbrio são proporcionais ao valor de mercado de cada empresa.

`Ticker.info["marketCap"]` devolve **só o valor de hoje**. Para o toy example isso basta.

In [ ]:
market_cap = pd.Series({t: yf.Ticker(t + ".SA").info["marketCap"] for t in TICKERS})

w_mkt = market_cap / market_cap.sum()

pd.DataFrame({"market_cap (R$ bi)": market_cap / 1e9, "w_mkt": w_mkt}).round(3)

### 1.3 CDI (taxa livre de risco)

API SGS do Banco Central, série 12 = CDI diário **em % ao dia**.

- As datas vêm como `dd/mm/aaaa` e os valores como texto.
- A taxa vem em %, então é preciso dividir por 100 para obter decimal.

In [ ]:
url = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados"
params = {
    "formato": "json",
    "dataInicial": pd.Timestamp(INICIO).strftime("%d/%m/%Y"),
    "dataFinal": pd.Timestamp(FIM).strftime("%d/%m/%Y"),
}
resp = requests.get(url, params=params, timeout=30)
resp.raise_for_status()

cdi = pd.DataFrame(resp.json())
cdi["data"] = pd.to_datetime(cdi["data"], dayfirst=True)
cdi = cdi.set_index("data")["valor"].astype(float) / 100   # % a.d. -> decimal a.d.

# taxa anualizada (252 dias úteis), só para conferir a ordem de grandeza
print(f"CDI último dia: {cdi.iloc[-1]:.6f} a.d. = {(1 + cdi.iloc[-1])**252 - 1:.2%} a.a.")
cdi.tail()

### Checagem antes de seguir

- Os preços têm datas parecidas com as do CDI (os dois usam dias úteis)?
- Os pesos `w_mkt` somam 1?
- O CDI anualizado está perto da Selic atual?

In [ ]:
print("w_mkt soma:", w_mkt.sum())
print("dias de preço:", len(precos), "| dias de CDI:", len(cdi))
print("datas de preço sem CDI:", len(precos.index.difference(cdi.index)))

## Etapa 2: Retornos e covariância $\Sigma$

### 2.1 Retornos diários em excesso

O modelo trabalha com **retorno em excesso**: quanto o ativo rendeu **acima do CDI**. Ninguém precisa correr risco para ganhar o CDI, então o prêmio pelo risco é o que sobra.

$$r^{exc}_{i,t} = \frac{P_{i,t}}{P_{i,t-1}} - 1 - r_{f,t}$$

Usamos retorno **simples**, e não logarítmico, porque o retorno de uma carteira é a média ponderada dos retornos simples dos ativos: $r_p = w^\top r$. O Black-Litterman e o Markowitz dependem dessa linearidade.

In [ ]:
retornos = precos.pct_change().dropna()

# alinha o CDI às datas de pregão e subtrai linha a linha
excesso = retornos.sub(cdi.reindex(retornos.index), axis=0)

print("NaN após alinhar:", excesso.isna().sum().sum())
excesso.head()

### 2.2 Anualização

Os dados são diários, mas views e resultados são mais fáceis de ler **ao ano**. Supondo retornos independentes entre os dias:

- média anual $\approx 252 \times$ média diária
- covariância anual $\approx 252 \times$ covariância diária, logo a volatilidade escala com $\sqrt{252}$

$\Sigma$ é a peça mais importante do modelo: aparece no prior, no posterior e na otimização.

In [ ]:
DU = 252   # dias úteis no ano

Sigma = excesso.cov() * DU
vol = np.sqrt(np.diag(Sigma))

print("Σ anual:")
display(Sigma.round(4))
print("Volatilidade anual:", dict(zip(TICKERS, vol.round(3))))
print("Correlação:")
excesso.corr().round(2)

### 2.3 Por que não usar a média histórica como retorno esperado?

Essa é a motivação do Black-Litterman. Veja a média histórica do excesso de retorno de cada ativo e o que o Markowitz faria com ela, usando os pesos sem restrição $w = (\delta\Sigma)^{-1}\mu$:

In [ ]:
mu_hist = excesso.mean() * DU

w_hist = np.linalg.solve(2.5 * Sigma, mu_hist)   # δ = 2.5, explicado na etapa 3
w_hist = w_hist / w_hist.sum()                    # normaliza para somar 1

pd.DataFrame({"mu_hist": mu_hist, "w_markowitz_hist": w_hist}).round(3)

Repare que **cinco anos de dados definem a carteira**: a PETR4 subiu muito no período e a VALE3 ficou abaixo do CDI. O otimizador acredita totalmente nessas médias, por isso a carteira concentra em PETR4 e zera ou vende a VALE3.

As médias históricas são muito ruidosas. Com volatilidade de 30% ao ano, o erro-padrão da média em 5 anos é de cerca de $0{,}30/\sqrt{5} \approx 13\%$ ao ano, quase do tamanho da própria média. O Black-Litterman troca essa média por um ponto de partida mais estável: **o retorno que o mercado está implicitamente precificando**.

---

## Etapa 3: Prior (retornos de equilíbrio $\Pi$)

### 3.1 A ideia: otimização reversa

Um investidor de Markowitz com aversão ao risco $\delta$ escolhe os pesos

$$w^* = (\delta\Sigma)^{-1}\mu.$$

Se **o mercado inteiro** segue essa regra e o resultado observado é $w_{mkt}$, basta isolar $\mu$ para obter os retornos que justificam os pesos atuais:

$$\boxed{\Pi = \delta\,\Sigma\,w_{mkt}}$$

Esse $\Pi$ é o **prior**: o que esperar de cada ativo quando não há opinião própria.

### 3.2 Aversão ao risco $\delta$

$\delta$ é o prêmio de risco que o mercado exige por unidade de variância:

$$\delta = \frac{E[r_m] - r_f}{\sigma_m^2}$$

Vamos estimá-lo de duas formas: pela carteira de mercado dos 3 ativos e pelo Ibovespa.

In [ ]:
# (a) carteira de mercado dos 3 ativos
exc_mkt = excesso @ w_mkt
delta_3 = (exc_mkt.mean() * DU) / (exc_mkt.var() * DU)

# (b) Ibovespa
ibov = yf.download("^BVSP", start=INICIO, end=FIM, auto_adjust=True, progress=False)["Close"].squeeze()
exc_ibov = ibov.pct_change().dropna() - cdi.reindex(ibov.index).dropna()
exc_ibov = exc_ibov.dropna()
delta_ibov = (exc_ibov.mean() * DU) / (exc_ibov.var() * DU)

print(f"δ (3 ativos):  {delta_3:.2f}")
print(f"δ (Ibovespa):  {delta_ibov:.2f}")

**O δ pelo Ibovespa sai negativo**, porque no período o Ibovespa rendeu menos que o CDI. Um δ negativo significaria investidores que *gostam* de risco, o que não faz sentido como equilíbrio. Já o δ dos 3 ativos é alto porque PETR4 teve um período excepcional.

Isso mostra que $\delta$ estimado pela história herda o mesmo ruído das médias. Por isso a prática comum é **fixar** um valor razoável. A literatura (He & Litterman, 1999; Idzorek, 2005) usa algo entre 2 e 3. Vamos usar **δ = 2,5** e discutir a sensibilidade depois.

In [ ]:
DELTA = 2.5

Pi = DELTA * Sigma @ w_mkt

pd.DataFrame({
    "w_mkt": w_mkt,
    "Pi (prior)": Pi,
    "mu_hist": mu_hist,
}).round(3)

### 3.3 Checagem: a otimização reversa recupera $w_{mkt}$

Se aplicarmos o Markowitz ao $\Pi$, os pesos ótimos devem ser **exatamente** os de mercado. Sem views, o modelo diz para ficar com o mercado.

In [ ]:
w_check = np.linalg.solve(DELTA * Sigma, Pi)

pd.DataFrame({"w_mkt": w_mkt, "w_de_Pi": w_check}).round(4)

### Para discutir antes da Etapa 4

1. Compare `Pi` com `mu_hist`. O prior é mais "comportado"? Qual ativo tem o maior $\Pi$ e por quê? (Pense em peso de mercado × volatilidade × correlação.)
2. O que acontece com $\Pi$ se δ dobrar? E com os pesos ótimos? (Teste e veja que os pesos não mudam.)
3. **Views**: que opiniões vocês têm sobre PETR4, VALE3 e ITUB4 para os próximos 12 meses? Formulem pelo menos uma **absoluta** (ex.: "ITUB4 vai render 8% acima do CDI") e uma **relativa** (ex.: "VALE3 vai superar PETR4 em 3%"), cada uma com um grau de confiança.

---

## Etapa 4: Views ($P$, $Q$, $\Omega$, $\tau$)

Toda view é escrita como **uma combinação linear dos retornos que vale um número**:

$$P\,\mu = Q + \varepsilon, \qquad \varepsilon \sim N(0, \Omega)$$

- **$P$** ($k \times n$): uma linha por view, uma coluna por ativo, indicando *quais* ativos a view envolve.
- **$Q$** ($k \times 1$): *quanto* a view diz que a combinação vai render (excesso ao CDI, ao ano).
- **$\Omega$** ($k \times k$, diagonal): a *incerteza* de cada view. Quanto maior, menos confiança.

### Views de exemplo (troquem pelas de vocês)

| # | Tipo | View | Confiança |
|---|---|---|---|
| 1 | Absoluta | ITUB4 vai render **10%** a.a. acima do CDI | 60% |
| 2 | Relativa | VALE3 vai superar PETR4 em **3%** a.a. | 40% |

Para comparar, o prior diz ITUB4 = 8,2% e VALE3 − PETR4 = −6,6%. As duas views discordam do mercado, a segunda bastante.

In [ ]:
# colunas na ordem de TICKERS: PETR4, VALE3, ITUB4
P = np.array([
    [ 0, 0, 1],    # view 1: ITUB4
    [-1, 1, 0],    # view 2: VALE3 - PETR4  (linha soma zero = view relativa)
])
Q = np.array([0.10, 0.03])
confianca = np.array([0.60, 0.40])

print("O que o prior diz sobre cada view (P @ Pi):", (P @ Pi).round(4))
print("O que as views dizem (Q):                  ", Q)

### $\tau$: a incerteza do prior

O mercado também não sabe o retorno esperado verdadeiro. O Black-Litterman trata $\Pi$ como uma estimativa com covariância $\tau\Sigma$:

$$\mu \sim N(\Pi,\ \tau\Sigma)$$

$\tau$ pequeno significa confiar muito no prior. O valor usual é **$\tau = 0{,}05$**, próximo de $1/T$ com $T \approx 20$ anos de dados.

### $\Omega$ a partir da confiança

$\Omega$ é a parte mais subjetiva do modelo. Uma forma prática de traduzir "confiança $c$" em variância é comparar com a incerteza que o próprio prior tem sobre a view, $p_k\,\tau\Sigma\,p_k^\top$:

$$\Omega_{kk} = \frac{1-c_k}{c_k}\; p_k\,\tau\Sigma\,p_k^\top$$

- $c = 50\%$: a view tem a mesma incerteza do prior, e o resultado fica no meio do caminho.
- $c \to 100\%$: $\Omega \to 0$, e o posterior obedece exatamente a view.
- $c \to 0\%$: $\Omega \to \infty$, e a view é ignorada.

(É uma aproximação simplificada da ideia de Idzorek, 2005. O Ω de He & Litterman, $\mathrm{diag}(P\,\tau\Sigma\,P^\top)$, equivale a $c = 50\%$ em todas as views.)

In [ ]:
TAU = 0.05

tauSigma = TAU * Sigma.values
var_prior_views = np.diag(P @ tauSigma @ P.T)          # incerteza do prior sobre cada view
Omega = np.diag((1 - confianca) / confianca * var_prior_views)

pd.DataFrame({
    "var. prior da view": var_prior_views,
    "Omega (var. da view)": np.diag(Omega),
    "desvio da view": np.sqrt(np.diag(Omega)),
}, index=["view 1", "view 2"]).round(5)

---

## Etapa 5: Posterior ($\mu_{BL}$)

Agora há duas fontes de informação sobre $\mu$:

- **Prior (mercado):** $\mu \sim N(\Pi,\ \tau\Sigma)$
- **Views (vocês):** $P\mu \sim N(Q,\ \Omega)$

O teorema de Bayes combina as duas numa **média ponderada pelas precisões** (inverso da variância):

$$\mu_{BL} = \big[(\tau\Sigma)^{-1} + P^\top\Omega^{-1}P\big]^{-1}\big[(\tau\Sigma)^{-1}\Pi + P^\top\Omega^{-1}Q\big]$$

É a mesma ideia de juntar duas medições ruidosas de uma grandeza: a mais precisa pesa mais. A incerteza que resta sobre $\mu$ e a covariância usada na otimização são

$$M = \big[(\tau\Sigma)^{-1} + P^\top\Omega^{-1}P\big]^{-1}, \qquad \Sigma_{BL} = \Sigma + M$$

$\Sigma_{BL}$ soma o risco do retorno ($\Sigma$) e a incerteza que ainda resta sobre a média ($M$).

In [ ]:
inv = np.linalg.inv

prec_prior = inv(tauSigma)                 # (τΣ)^-1
prec_views = P.T @ inv(Omega) @ P          # P' Ω^-1 P

M = inv(prec_prior + prec_views)
mu_BL = M @ (prec_prior @ Pi.values + P.T @ inv(Omega) @ Q)
mu_BL = pd.Series(mu_BL, index=TICKERS)

Sigma_BL = Sigma + M

pd.DataFrame({"Pi (prior)": Pi, "mu_BL (posterior)": mu_BL, "diferença": mu_BL - Pi}).round(4)

### Checagem: o posterior fica entre o prior e as views

Para cada view, compare o que o prior dizia ($P\Pi$), o que vocês disseram ($Q$) e o resultado ($P\mu_{BL}$). O posterior deve ficar **entre os dois**, mais perto de $Q$ quanto maior a confiança.

Repare também que todos os ativos mudam, mesmo os que não aparecem numa view. Isso vem das correlações em $\Sigma$: a informação de uma view se espalha pelos ativos correlacionados.

In [ ]:
pd.DataFrame({
    "prior  P·Pi": P @ Pi.values,
    "view   Q": Q,
    "post.  P·mu_BL": P @ mu_BL.values,
    "confiança": confianca,
}, index=["view 1: ITUB4", "view 2: VALE3-PETR4"]).round(4)

---

## Etapa 6: Otimização da carteira

Com $\mu_{BL}$ e $\Sigma_{BL}$ em mãos, voltamos ao Markowitz, agora com retornos que misturam mercado e opinião.

O investidor maximiza a utilidade média-variância:

$$\max_w\; w^\top\mu_{BL} - \frac{\delta}{2}\,w^\top\Sigma_{BL}\,w$$

### 6.1 Sem restrições (solução analítica)

$$w^* = (\delta\,\Sigma_{BL})^{-1}\mu_{BL}$$

Sem restrição de soma, o que sobra de $1 - \sum w$ fica no CDI (ou é tomado emprestado, se negativo).

In [ ]:
w_irrestrito = pd.Series(np.linalg.solve(DELTA * Sigma_BL, mu_BL), index=TICKERS)

print(f"Soma dos pesos em ações: {w_irrestrito.sum():.3f}  (resto no CDI: {1 - w_irrestrito.sum():.3f})")
pd.DataFrame({"w_mkt": w_mkt, "w_BL irrestrito": w_irrestrito, "desvio": w_irrestrito - w_mkt}).round(3)

Compare com o Markowitz pela média histórica da Etapa 2.3: lá a carteira ia a 110% em PETR4 e vendia VALE3. Aqui os pesos ficam **perto do mercado e se movem na direção das views**:
- ITUB4 sobe (view 1 otimista);
- VALE3 sobe e PETR4 cai (view 2).

Essa é a grande vantagem do Black-Litterman: **as views movem a carteira de forma intuitiva**, sem o otimizador concentrar em tudo.

A soma dos pesos deixou de ser exatamente 1. Duas forças atuam:
- as views elevam o retorno esperado médio (ITUB4 e VALE3 sobem mais do que PETR4 cai), o que puxa a exposição a ações **para cima**;
- $\Sigma_{BL} > \Sigma$: a incerteza extra sobre a média ($M$) deixa o investidor mais conservador, puxando **para baixo**.

Aqui a primeira força vence, e a carteira toma ~4% emprestado ao CDI.

### 6.2 Com restrições: long-only e totalmente investida

Na prática a carteira costuma ter $\sum w = 1$ e $w \ge 0$, sem venda a descoberto. Aí não há fórmula fechada e usamos um otimizador numérico.

In [ ]:
from scipy.optimize import minimize

def otimiza(mu, Sigma, delta=DELTA):
    """Max w'mu - δ/2 w'Σw  s.a.  soma(w)=1, w>=0."""
    mu, S = np.asarray(mu), np.asarray(Sigma)
    n = len(mu)
    res = minimize(
        lambda w: -(w @ mu - delta / 2 * w @ S @ w),   # minimiza o negativo
        x0=np.ones(n) / n,
        bounds=[(0, 1)] * n,
        constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
        method="SLSQP",
    )
    assert res.success, res.message
    return res.x

w_BL = pd.Series(otimiza(mu_BL, Sigma_BL), index=TICKERS)
w_hist_lo = pd.Series(otimiza(mu_hist, Sigma), index=TICKERS)   # Markowitz "ingênuo", só para comparar

resultado = pd.DataFrame({
    "w_mkt (prior)": w_mkt,
    "w_BL (posterior)": w_BL,
    "w_Markowitz hist.": w_hist_lo,
})
resultado.round(3)

In [ ]:
import matplotlib.pyplot as plt

ax = resultado.plot.bar(figsize=(8, 4), rot=0)
ax.set_ylabel("peso na carteira")
ax.set_title("Mercado × Black-Litterman × Markowitz com média histórica")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.tight_layout()
plt.show()

### 6.3 Retorno e risco esperados das carteiras

In [ ]:
def resumo(w, mu, S):
    ret = w @ mu
    vol = np.sqrt(w @ S @ w)
    return pd.Series({"retorno exc. (a.a.)": ret, "vol (a.a.)": vol, "Sharpe": ret / vol})

pd.DataFrame({
    "mercado": resumo(w_mkt.values, mu_BL.values, Sigma_BL.values),
    "Black-Litterman": resumo(w_BL.values, mu_BL.values, Sigma_BL.values),
}).round(3)

Os dois resumos usam as **mesmas** expectativas ($\mu_{BL}$, $\Sigma_{BL}$). É a comparação justa: *dado o que acreditamos*, a carteira BL tem retorno um pouco menor, mas risco menor ainda, e por isso Sharpe e utilidade maiores que a de mercado. Se as views estiverem erradas, a vantagem não se realiza, e é isso que o backtest do projeto vai testar.

---

## Etapa 7 (extra): sensibilidade à confiança

Como os pesos respondem à confiança na view 1? Mantemos a view 2 fixa e variamos $c_1$ de 1% a 99%.

In [ ]:
def black_litterman(Pi, Sigma, P, Q, confianca, tau=TAU):
    """Devolve (mu_BL, Sigma_BL). Junta as etapas 4 e 5 numa função."""
    tS = tau * np.asarray(Sigma)
    Om = np.diag((1 - confianca) / confianca * np.diag(P @ tS @ P.T))
    M = inv(inv(tS) + P.T @ inv(Om) @ P)
    mu = M @ (inv(tS) @ np.asarray(Pi) + P.T @ inv(Om) @ Q)
    return mu, np.asarray(Sigma) + M

grade = np.linspace(0.01, 0.99, 50)
pesos = []
for c1 in grade:
    mu_c, S_c = black_litterman(Pi, Sigma, P, Q, np.array([c1, confianca[1]]))
    pesos.append(otimiza(mu_c, S_c))
pesos = pd.DataFrame(pesos, index=grade, columns=TICKERS)

ax = pesos.plot(figsize=(8, 4))
ax.set_xlabel("confiança na view 1 (ITUB4 = 10%)")
ax.set_ylabel("peso (long-only)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.tight_layout()
plt.show()

### Resumo do fluxo

$$\underbrace{\text{preços, mkt cap, CDI}}_{1} \to \underbrace{\Sigma}_{2} \to \underbrace{\Pi = \delta\Sigma w_{mkt}}_{3:\ \text{prior}} \xrightarrow{\;P,\,Q,\,\Omega,\,\tau\ (4)\;} \underbrace{\mu_{BL},\ \Sigma_{BL}}_{5:\ \text{posterior}} \to \underbrace{w_{BL}}_{6:\ \text{otimização}}$$

### Para explorar com seu colega

1. Troquem as views de exemplo pelas de vocês e vejam como $w_{BL}$ muda.
2. Coloquem uma view **igual ao prior** (ex.: ITUB4 = 8,2%). Os pesos devem voltar para perto de $w_{mkt}$. Por quê?
3. Mudem $\tau$ para 0,01 e para 0,5. Com $\Omega$ definido pela confiança, $\mu_{BL}$ não muda. Por quê? (Dica: $\Omega$ também é proporcional a $\tau$.) O que muda é $\Sigma_{BL}$.
4. Quais dessas funções viram `bl/core.py` no projeto? (`black_litterman`, `otimiza` e a otimização reversa são boas candidatas.)